#02 — Exploratory Dataset Analysis (EDA)
## Relational multilabel modeling with clinical co-occurrence

**Purpose of this notebook:**
1. Integrity audit (duplicates, orphaned images, missing values)
2. Label distribution and imbalance
3. Analysis of co-occurrence between labels (frequency, PMI, phi-coefficient, lift)
4. Bilingual Views (PT/EN)
5. Consolidation of results in JSON

**Outputs:**
- `project/results/02_eda_stats.json` — consolidated statistics
- `project/figs/` — graphics in PT and EN

In [ ]:
import os
import json
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from tqdm import tqdm

# Paths
ROOT = Path(r"/workspace")
DATA_DIR = ROOT / "Dev" / "Data"
IMGS_DIR = DATA_DIR / "Imgs"
CSV_PATH = DATA_DIR / "Imgs-anotadas" / "dataset_labels.csv"

OUT_DIR = ROOT / 'project'
SPLITS_DIR = OUT_DIR / "splits"
RESULTS_DIR = OUT_DIR / "results"
FIGS_DIR = OUT_DIR / "figs"

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Images dir: {IMGS_DIR}")
print(f"CSV path:   {CSV_PATH}")
print(f"Output:     {OUT_DIR}")

## 1. Loading and initial inspection

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

In [ ]:
# Rename columns for easier handling
df = df_raw.copy()
df.rename(columns={"Coluna 1": "image_name"}, inplace=True)

# Label columns
LABEL_COLS = ["NORMAL", "ALTERADO", "SALIVA", "LUZ", "ENANTEMA",
              "PÓLIPO", "ÚLCERA", "EROSÃO", "MICRONODULARIDADE",
              "ECTASIA VASCULAR", "NEOPLASIA"]

# English label names for bilingual plots
LABEL_EN = {
    "NORMAL": "NORMAL",
    "ALTERADO": "ALTERED",
    "SALIVA": "SALIVA",
    "LUZ": "LIGHT REFLECTION",
    "ENANTEMA": "ENANTHEMA",
    "PÓLIPO": "POLYP",
    "ÚLCERA": "ULCER",
    "EROSÃO": "EROSION",
    "MICRONODULARIDADE": "MICRONODULARITY",
    "ECTASIA VASCULAR": "VASCULAR ECTASIA",
    "NEOPLASIA": "NEOPLASIA"
}

print(f"Total rows: {len(df)}")
print(f"Label columns: {len(LABEL_COLS)}")

## 2. Integrity audit

In [ ]:
# 2.1 List all image files on disk
img_files_on_disk = set(f.name for f in IMGS_DIR.iterdir() if f.suffix.lower() in (".jpg", ".jpeg", ".png"))
print(f"Image files on disk: {len(img_files_on_disk)}")

# 2.2 Image names in CSV
img_names_csv = set(df["image_name"].dropna().astype(str))
print(f"Image names in CSV:  {len(img_names_csv)}")

# 2.3 Orphan images (on disk but not in CSV)
orphan_imgs = img_files_on_disk - img_names_csv
print(f"\nOrphan images (on disk, not in CSV): {len(orphan_imgs)}")
if len(orphan_imgs) <= 20:
    print(f"  Files: {sorted(orphan_imgs)}")

# 2.4 Missing images (in CSV but not on disk)
missing_imgs = img_names_csv - img_files_on_disk
print(f"\nMissing images (in CSV, not on disk): {len(missing_imgs)}")
if len(missing_imgs) <= 20:
    print(f"  Names: {sorted(missing_imgs)[:20]}")

In [ ]:
# 2.5 Duplicate image_name in CSV
dup_mask = df["image_name"].duplicated(keep=False)
dup_names = df.loc[dup_mask, "image_name"].unique()
print(f"Duplicated image_name entries: {len(dup_names)} unique names, {dup_mask.sum()} rows total")
if len(dup_names) > 0:
    print(f"  Names: {list(dup_names[:20])}")
    display(df[dup_mask].sort_values("image_name").head(20))

In [ ]:
# 2.6 Missing values ​​per label column
print("Missing values (NaN) per label column:")
missing_per_col = df[LABEL_COLS].isna().sum()
print(missing_per_col[missing_per_col > 0].to_string())
if missing_per_col.sum() == 0:
    print("  (none)")

# 2.7 Unique values ​​per label column (expect {1, 2} or {1, 2, NaN})
print("\nUnique values per label column:")
for col in LABEL_COLS:
    vals = sorted(df[col].dropna().unique())
    print(f"  {col}: {vals}")

In [ ]:
# 2.8 Filter to usable rows: image exists on disk + no NaN image_name
df["has_file"] = df["image_name"].apply(lambda x: str(x) in img_files_on_disk if pd.notna(x) else False)
df_valid = df[df["has_file"]].copy()
print(f"Rows with corresponding image file: {len(df_valid)} / {len(df)}")

# 2.9 Convert label encoding: 1=present, 2=absent -> 1/0; NaN stays NaN
for col in LABEL_COLS:
    df_valid[col] = df_valid[col].map({1: 1, 2: 0})

print(f"\nAfter conversion (1=present, 0=absent):")
df_valid[LABEL_COLS].head()

In [ ]:
# 2.10 Handle duplicates: OR logic — union of all annotations across duplicate rows.
# Safer for medical data: if any annotator marked a pathology, it is kept.
# Consistent with 02_splits_baseline deduplication rule.
if len(dup_names) > 0:
    def merge_dups(group):
        merged = group.iloc[0].copy()
        merged[LABEL_COLS] = group[LABEL_COLS].max()  # OR: union of all findings
        return merged

    df_dups    = df_valid[df_valid["image_name"].isin(dup_names)]
    df_no_dups = df_valid[~df_valid["image_name"].isin(dup_names)]

    df_best_dups = (
        df_dups
        .groupby("image_name", group_keys=False)
        .apply(merge_dups)
        .reset_index(drop=True)
    )

    df_clean = pd.concat([df_no_dups, df_best_dups], ignore_index=True)
    print(f"After OR deduplication: {len(df_clean)} rows (removed {len(df_valid) - len(df_clean)} duplicate rows)")
else:
    df_clean = df_valid.copy()
    print("No duplicates to remove.")

print(f"\nFinal clean dataset: {len(df_clean)} images")

In [ ]:
# 2.11 Dataset version hash (for reproducibility)
csv_bytes = df_clean[["image_name"] + LABEL_COLS].to_csv(index=False).encode("utf-8")
dataset_hash = hashlib.sha256(csv_bytes).hexdigest()[:16]
print(f"Dataset version hash: {dataset_hash}")
print(f"N images: {len(df_clean)}, N labels: {len(LABEL_COLS)}")

## 3. Label distribution and imbalance

In [ ]:
# 3.1 Label prevalence table
N = len(df_clean)
label_stats = []
for col in LABEL_COLS:
    present = df_clean[col].sum()
    absent = (df_clean[col] == 0).sum()
    missing = df_clean[col].isna().sum()
    prevalence = present / (present + absent) if (present + absent) > 0 else 0
    imbalance_ratio = absent / present if present > 0 else np.inf
    label_stats.append({
        "label_pt": col,
        "label_en": LABEL_EN[col],
        "present": int(present),
        "absent": int(absent),
        "missing": int(missing),
        "prevalence": round(prevalence, 4),
        "imbalance_ratio": round(imbalance_ratio, 2)
    })

df_stats = pd.DataFrame(label_stats)
df_stats = df_stats.sort_values("present", ascending=False).reset_index(drop=True)
print("Label distribution:")
df_stats

In [ ]:
# 3.2 Bar plot: Label prevalence (PT)
fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#2ecc71" if row["imbalance_ratio"] < 5 else "#f39c12" if row["imbalance_ratio"] < 50 else "#e74c3c" 
          for _, row in df_stats.iterrows()]
ax.barh(df_stats["label_pt"], df_stats["present"], color=colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Número de imagens com rótulo presente", fontsize=12)
ax.set_title("Distribuição de Rótulos — Gastroscopy Dataset", fontsize=14, fontweight="bold")
ax.invert_yaxis()
for i, row in df_stats.iterrows():
    ax.text(row["present"] + 5, i, f"{row['present']} ({row['prevalence']*100:.1f}%)", va="center", fontsize=9)
ax.set_xlim(0, df_stats["present"].max() * 1.25)
plt.tight_layout()
plt.savefig(FIGS_DIR / "01_label_distribution_PT.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGS_DIR / '01_label_distribution_PT.png'}")

In [ ]:
# 3.3 Bar plot: Label prevalence (EN)
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(df_stats["label_en"], df_stats["present"], color=colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Number of images with label present", fontsize=12)
ax.set_title("Label Distribution — Gastroscopy Dataset", fontsize=14, fontweight="bold")
ax.invert_yaxis()
for i, row in df_stats.iterrows():
    ax.text(row["present"] + 5, i, f"{row['present']} ({row['prevalence']*100:.1f}%)", va="center", fontsize=9)
ax.set_xlim(0, df_stats["present"].max() * 1.25)
plt.tight_layout()
plt.savefig(FIGS_DIR / "01_label_distribution_EN.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGS_DIR / '01_label_distribution_EN.png'}")

## 4. Multilabel distribution (labels per image)

## 3.5. Integration with Visual Similarity Groups (Pseudo-Patients)
Integration of the mapping generated in the notebook `01_visual_similarity_groups.ipynb`.
This ensures that the exploratory analysis accurately reflects the groups.

In [ ]:
groups_path = SPLITS_DIR / "image_group_mapping.csv"

if groups_path.exists():
    df_groups = pd.read_csv(groups_path)
    df_clean = df_clean.merge(df_groups, on="image_name", how="inner")

    print("Integração de Grupos concluída.")
    print(f"Imagens válidas integradas: {len(df_clean)}")
    print(f"Total de pseudo-pacientes (grupos): {df_clean['group_id'].nunique()}")

else:
    df_clean["group_id"] = np.arange(len(df_clean))

    print("Arquivo image_group_mapping.csv não encontrado. Simulando grupos de tamanho 1.")

In [ ]:
# 4.1 Total positive labels per image (all 11 labels)
df_clean["n_labels_total"] = df_clean[LABEL_COLS].sum(axis=1)

# 4.2 Clinical pathology labels only (excluding NORMAL, ALTERED, SALIVA, LIGHT)
PATHOLOGY_COLS = ["ENANTEMA", "PÓLIPO", "ÚLCERA", "EROSÃO", "MICRONODULARIDADE",
                  "ECTASIA VASCULAR", "NEOPLASIA"]
ARTIFACT_COLS = ["SALIVA", "LUZ"]
META_COLS = ["NORMAL", "ALTERADO"]

df_clean["n_pathologies"] = df_clean[PATHOLOGY_COLS].sum(axis=1)
df_clean["n_artifacts"] = df_clean[ARTIFACT_COLS].sum(axis=1)

print("Distribution of total positive labels per image:")
print(df_clean["n_labels_total"].value_counts().sort_index())
print(f"\nDistribution of pathology labels per image:")
print(df_clean["n_pathologies"].value_counts().sort_index())

In [ ]:
# 4.3 Histogram: labels per image (PT)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All labels
counts_total = df_clean["n_labels_total"].value_counts().sort_index()
axes[0].bar(counts_total.index, counts_total.values, color="#3498db", edgecolor="black", linewidth=0.5)
axes[0].set_xlabel("Número de rótulos positivos", fontsize=11)
axes[0].set_ylabel("Número de imagens", fontsize=11)
axes[0].set_title("Rótulos positivos por imagem (todos)", fontsize=12, fontweight="bold")
for x, y in zip(counts_total.index, counts_total.values):
    axes[0].text(x, y + 5, str(y), ha="center", fontsize=8)

# Pathologies only
counts_path = df_clean["n_pathologies"].value_counts().sort_index()
axes[1].bar(counts_path.index, counts_path.values, color="#e74c3c", edgecolor="black", linewidth=0.5)
axes[1].set_xlabel("Número de patologias positivas", fontsize=11)
axes[1].set_ylabel("Número de imagens", fontsize=11)
axes[1].set_title("Patologias por imagem (sem artefatos/meta)", fontsize=12, fontweight="bold")
for x, y in zip(counts_path.index, counts_path.values):
    axes[1].text(x, y + 5, str(y), ha="center", fontsize=8)

plt.tight_layout()
plt.savefig(FIGS_DIR / "02_labels_per_image_PT.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 4.4 Histogram: labels per image (EN)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(counts_total.index, counts_total.values, color="#3498db", edgecolor="black", linewidth=0.5)
axes[0].set_xlabel("Number of positive labels", fontsize=11)
axes[0].set_ylabel("Number of images", fontsize=11)
axes[0].set_title("Positive labels per image (all)", fontsize=12, fontweight="bold")
for x, y in zip(counts_total.index, counts_total.values):
    axes[0].text(x, y + 5, str(y), ha="center", fontsize=8)

axes[1].bar(counts_path.index, counts_path.values, color="#e74c3c", edgecolor="black", linewidth=0.5)
axes[1].set_xlabel("Number of positive pathologies", fontsize=11)
axes[1].set_ylabel("Number of images", fontsize=11)
axes[1].set_title("Pathologies per image (excl. artifacts/meta)", fontsize=12, fontweight="bold")
for x, y in zip(counts_path.index, counts_path.values):
    axes[1].text(x, y + 5, str(y), ha="center", fontsize=8)

plt.tight_layout()
plt.savefig(FIGS_DIR / "02_labels_per_image_EN.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 4.5 Consistency check: NORMAL vs CHANGED
normal_only = ((df_clean["NORMAL"] == 1) & (df_clean["ALTERADO"] == 0)).sum()
alterado_only = ((df_clean["NORMAL"] == 0) & (df_clean["ALTERADO"] == 1)).sum()
both = ((df_clean["NORMAL"] == 1) & (df_clean["ALTERADO"] == 1)).sum()
neither = ((df_clean["NORMAL"] == 0) & (df_clean["ALTERADO"] == 0)).sum()

print("NORMAL vs ALTERADO consistency:")
print(f"  NORMAL only:   {normal_only}")
print(f"  ALTERADO only: {alterado_only}")
print(f"  Both = 1:      {both}")
print(f"  Neither = 1:   {neither}")

# ALTERED without any specific pathology
alterado_no_path = ((df_clean["ALTERADO"] == 1) & (df_clean[PATHOLOGY_COLS].sum(axis=1) == 0)).sum()
print(f"\n  ALTERADO=1 but no pathology label: {alterado_no_path}")
print(f"  -> These are 'altered' images with unspecified finding")

## 5. Co-occurrence analysis between labels

In [ ]:
# 5.1 Co-occurrence matrix (pathology + artifact labels)
ANALYSIS_COLS = PATHOLOGY_COLS + ARTIFACT_COLS  # 9 labels

def compute_cooccurrence_matrix(df, cols):
    """Raw co-occurrence count matrix."""
    n = len(cols)
    matrix = np.zeros((n, n), dtype=int)
    data = df[cols].values
    for i in range(n):
        for j in range(n):
            matrix[i, j] = int(((data[:, i] == 1) & (data[:, j] == 1)).sum())
    return pd.DataFrame(matrix, index=cols, columns=cols)

coocc_matrix = compute_cooccurrence_matrix(df_clean, ANALYSIS_COLS)
print("Co-occurrence matrix (raw counts):")
coocc_matrix

In [ ]:
# 5.2 Compute association metrics for all pairs
def compute_pair_metrics(df, cols):
    """Compute PMI, nPMI, phi-coefficient, lift, Jaccard for all label pairs."""
    N = len(df)
    results = []
    
    for i, j in combinations(range(len(cols)), 2):
        col_i, col_j = cols[i], cols[j]
        
        # Skip pairs where either column has too many NaNs
        valid_mask = df[col_i].notna() & df[col_j].notna()
        n_valid = valid_mask.sum()
        if n_valid == 0:
            continue
        
        yi = df.loc[valid_mask, col_i].values
        yj = df.loc[valid_mask, col_j].values
        
        # Counts
        n11 = int(((yi == 1) & (yj == 1)).sum())  # both present
        n10 = int(((yi == 1) & (yj == 0)).sum())  # i present, j absent
        n01 = int(((yi == 0) & (yj == 1)).sum())  # i absent, j present
        n00 = int(((yi == 0) & (yj == 0)).sum())  # both absent
        
        # Marginals
        pi = (n11 + n10) / n_valid
        pj = (n11 + n01) / n_valid
        pij = n11 / n_valid
        
        # PMI (with Laplace smoothing)
        alpha = 1
        pi_s = (n11 + n10 + alpha) / (n_valid + 2 * alpha)
        pj_s = (n11 + n01 + alpha) / (n_valid + 2 * alpha)
        pij_s = (n11 + alpha) / (n_valid + 4 * alpha)
        
        pmi = np.log2(pij_s / (pi_s * pj_s)) if (pi_s * pj_s) > 0 else 0
        npmi = pmi / (-np.log2(pij_s)) if pij_s > 0 and pij_s < 1 else 0
        
        # Phi coefficient (Matthews correlation)
        denom = np.sqrt((n11+n10)*(n11+n01)*(n00+n10)*(n00+n01))
        phi = (n11*n00 - n10*n01) / denom if denom > 0 else 0
        
        # Lift
        lift = pij / (pi * pj) if (pi * pj) > 0 else 0
        
        # Jaccard
        jaccard = n11 / (n11 + n10 + n01) if (n11 + n10 + n01) > 0 else 0
        
        results.append({
            "label_a_pt": col_i,
            "label_b_pt": col_j,
            "label_a_en": LABEL_EN[col_i],
            "label_b_en": LABEL_EN[col_j],
            "co_occurrence": n11,
            "n_valid": int(n_valid),
            "P(a)": round(pi, 4),
            "P(b)": round(pj, 4),
            "P(a,b)": round(pij, 4),
            "PMI": round(pmi, 4),
            "nPMI": round(npmi, 4),
            "phi": round(phi, 4),
            "lift": round(lift, 4),
            "jaccard": round(jaccard, 4)
        })
    
    return pd.DataFrame(results)

df_pairs = compute_pair_metrics(df_clean, ANALYSIS_COLS)
df_pairs = df_pairs.sort_values("co_occurrence", ascending=False).reset_index(drop=True)
print(f"Total pairs analyzed: {len(df_pairs)}")
df_pairs.head(15)

In [ ]:
# 5.3 Heatmap: Co-occurrence matrix (PT)
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(coocc_matrix, dtype=bool), k=0)
sns.heatmap(coocc_matrix, mask=mask, annot=True, fmt="d", cmap="YlOrRd",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Imagens"})
ax.set_title("Matriz de Co-ocorrência entre Rótulos", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR / "03_cooccurrence_matrix_PT.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 5.4 Heatmap: Co-occurrence matrix (EN)
coocc_matrix_en = coocc_matrix.copy()
coocc_matrix_en.index = [LABEL_EN[c] for c in coocc_matrix_en.index]
coocc_matrix_en.columns = [LABEL_EN[c] for c in coocc_matrix_en.columns]

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(coocc_matrix_en, dtype=bool), k=0)
sns.heatmap(coocc_matrix_en, mask=mask, annot=True, fmt="d", cmap="YlOrRd",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Images"})
ax.set_title("Label Co-occurrence Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR / "03_cooccurrence_matrix_EN.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 5.5 Heatmap: nPMI matrix (PT)
npmi_matrix = pd.DataFrame(0.0, index=ANALYSIS_COLS, columns=ANALYSIS_COLS)
for _, row in df_pairs.iterrows():
    npmi_matrix.loc[row["label_a_pt"], row["label_b_pt"]] = row["nPMI"]
    npmi_matrix.loc[row["label_b_pt"], row["label_a_pt"]] = row["nPMI"]

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(npmi_matrix, dtype=bool), k=0)
sns.heatmap(npmi_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={"label": "nPMI"})
ax.set_title("Associação entre Rótulos (nPMI normalizado)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR / "04_npmi_matrix_PT.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 5.6 Heatmap: nPMI matrix (EN)
npmi_matrix_en = npmi_matrix.copy()
npmi_matrix_en.index = [LABEL_EN[c] for c in npmi_matrix_en.index]
npmi_matrix_en.columns = [LABEL_EN[c] for c in npmi_matrix_en.columns]

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(npmi_matrix_en, dtype=bool), k=0)
sns.heatmap(npmi_matrix_en, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={"label": "nPMI"})
ax.set_title("Label Association (normalized PMI)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR / "04_npmi_matrix_EN.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 5.7 Phi coefficient matrix (PT)
phi_matrix = pd.DataFrame(0.0, index=ANALYSIS_COLS, columns=ANALYSIS_COLS)
for _, row in df_pairs.iterrows():
    phi_matrix.loc[row["label_a_pt"], row["label_b_pt"]] = row["phi"]
    phi_matrix.loc[row["label_b_pt"], row["label_a_pt"]] = row["phi"]
for lbl in ANALYSIS_COLS:
    phi_matrix.loc[lbl, lbl] = 1.0

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(phi_matrix, dtype=bool), k=0)
sns.heatmap(phi_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-0.5, vmax=0.5, linewidths=0.5, ax=ax,
            cbar_kws={"label": "Phi (Matthews)"})
ax.set_title("Correlação entre Rótulos (Phi / Matthews)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR / "05_phi_matrix_PT.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 5.8 Phi coefficient matrix (EN)
phi_matrix_en = phi_matrix.copy()
phi_matrix_en.index = [LABEL_EN[c] for c in phi_matrix_en.index]
phi_matrix_en.columns = [LABEL_EN[c] for c in phi_matrix_en.columns]

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(phi_matrix_en, dtype=bool), k=0)
sns.heatmap(phi_matrix_en, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-0.5, vmax=0.5, linewidths=0.5, ax=ax,
            cbar_kws={"label": "Phi (Matthews)"})
ax.set_title("Label Correlation (Phi / Matthews Coefficient)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR / "05_phi_matrix_EN.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Audit of suspicious pairs (co-occurrence = 4)

In [ ]:
# 6.1 Identify images responsible for pairs with co_occurrence = 4
# These likely come from a few heavily-labeled images
suspect_pairs = df_pairs[df_pairs["co_occurrence"] == 4][["label_a_pt", "label_b_pt"]]
print(f"Pairs with co_occurrence = 4: {len(suspect_pairs)}")
print()

# Find images with many pathology labels (likely responsible)
high_label_imgs = df_clean[df_clean["n_pathologies"] >= 4].copy()
print(f"Images with >= 4 pathologies: {len(high_label_imgs)}")
if len(high_label_imgs) > 0:
    print("\nThese images and their labels:")
    display(high_label_imgs[["image_name"] + PATHOLOGY_COLS + ["n_pathologies"]].sort_values("n_pathologies", ascending=False))

In [ ]:
# 6.2 For the extreme cases (>=6 pathologies), check if they inflate co-occurrence
# This is important: if 4 images carry ALL rare labels, then many pair counts = 4
# are not independent clinical co-occurrences but a single annotation artifact

extreme_imgs = df_clean[df_clean["n_pathologies"] >= 6]
if len(extreme_imgs) > 0:
    print(f"ALERT: {len(extreme_imgs)} images with 6+ pathologies.")
    print("These images are likely responsible for inflating co-occurrence counts.")
    print("Recommendation: flag for manual verification by clinician.\n")
    
    # Show which pairs are entirely explained by these extreme images
    for _, img_row in extreme_imgs.iterrows():
        active = [c for c in PATHOLOGY_COLS if img_row[c] == 1]
        print(f"  {img_row['image_name']}: {active}")
else:
    print("No extreme images (6+ pathologies) found.")

## 7. Artifact × pathology interaction

In [ ]:
# 7.1 Artifact x Pathology interaction table
artifact_path_pairs = []
for art in ARTIFACT_COLS:
    for path in PATHOLOGY_COLS:
        n_both = int(((df_clean[art] == 1) & (df_clean[path] == 1)).sum())
        n_path = int((df_clean[path] == 1).sum())
        pct = (n_both / n_path * 100) if n_path > 0 else 0
        artifact_path_pairs.append({
            "artifact_pt": art,
            "artifact_en": LABEL_EN[art],
            "pathology_pt": path,
            "pathology_en": LABEL_EN[path],
            "co_occurrence": n_both,
            "n_pathology": n_path,
            "pct_pathology_with_artifact": round(pct, 1)
        })

df_art_path = pd.DataFrame(artifact_path_pairs)
df_art_path = df_art_path.sort_values("co_occurrence", ascending=False).reset_index(drop=True)
print("Artifact x Pathology interaction (top 10):")
df_art_path.head(10)

In [ ]:
# 7.2 Stacked bar: % of each pathology with/without artifact (PT)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, art in enumerate(ARTIFACT_COLS):
    path_names = []
    pcts_with = []
    pcts_without = []
    for path in PATHOLOGY_COLS:
        n_path = int((df_clean[path] == 1).sum())
        if n_path == 0:
            continue
        n_both = int(((df_clean[art] == 1) & (df_clean[path] == 1)).sum())
        pct = n_both / n_path * 100
        path_names.append(path)
        pcts_with.append(pct)
        pcts_without.append(100 - pct)
    
    axes[idx].barh(path_names, pcts_with, color="#e74c3c", label=f"Com {art}")
    axes[idx].barh(path_names, pcts_without, left=pcts_with, color="#2ecc71", label=f"Sem {art}")
    axes[idx].set_xlabel("%", fontsize=11)
    axes[idx].set_title(f"Patologias com/sem {art}", fontsize=12, fontweight="bold")
    axes[idx].legend(loc="lower right")
    axes[idx].set_xlim(0, 100)

plt.tight_layout()
plt.savefig(FIGS_DIR / "06_artifact_pathology_PT.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 7.3 Stacked bar: % of each pathology with/without artifact (EN)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, art in enumerate(ARTIFACT_COLS):
    path_names_en = []
    pcts_with = []
    pcts_without = []
    for path in PATHOLOGY_COLS:
        n_path = int((df_clean[path] == 1).sum())
        if n_path == 0:
            continue
        n_both = int(((df_clean[art] == 1) & (df_clean[path] == 1)).sum())
        pct = n_both / n_path * 100
        path_names_en.append(LABEL_EN[path])
        pcts_with.append(pct)
        pcts_without.append(100 - pct)
    
    axes[idx].barh(path_names_en, pcts_with, color="#e74c3c", label=f"With {LABEL_EN[art]}")
    axes[idx].barh(path_names_en, pcts_without, left=pcts_with, color="#2ecc71", label=f"Without {LABEL_EN[art]}")
    axes[idx].set_xlabel("%", fontsize=11)
    axes[idx].set_title(f"Pathologies with/without {LABEL_EN[art]}", fontsize=12, fontweight="bold")
    axes[idx].legend(loc="lower right")
    axes[idx].set_xlim(0, 100)

plt.tight_layout()
plt.savefig(FIGS_DIR / "06_artifact_pathology_EN.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Imbalance Ratio and visualization

In [ ]:
# 8.1 Imbalance ratio plot (PT) - log scale
fig, ax = plt.subplots(figsize=(10, 6))
df_ir = df_stats[df_stats["label_pt"].isin(ANALYSIS_COLS)].sort_values("imbalance_ratio")
colors_ir = ["#2ecc71" if ir < 5 else "#f39c12" if ir < 50 else "#e74c3c" for ir in df_ir["imbalance_ratio"]]
ax.barh(df_ir["label_pt"], df_ir["imbalance_ratio"], color=colors_ir, edgecolor="black", linewidth=0.5)
ax.set_xscale("log")
ax.axvline(x=5, color="gray", linestyle="--", alpha=0.7, label="IR=5 (moderado)")
ax.axvline(x=50, color="gray", linestyle=":", alpha=0.7, label="IR=50 (severo)")
ax.set_xlabel("Imbalance Ratio (log scale)", fontsize=12)
ax.set_title("Razão de Desbalanceamento por Rótulo", fontsize=14, fontweight="bold")
ax.legend()
for i, (_, row) in enumerate(df_ir.iterrows()):
    ax.text(row["imbalance_ratio"] * 1.1, i, f"{row['imbalance_ratio']:.1f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIGS_DIR / "07_imbalance_ratio_PT.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 8.2 Imbalance ratio plot (EN) - log scale
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_ir["label_en"], df_ir["imbalance_ratio"], color=colors_ir, edgecolor="black", linewidth=0.5)
ax.set_xscale("log")
ax.axvline(x=5, color="gray", linestyle="--", alpha=0.7, label="IR=5 (moderate)")
ax.axvline(x=50, color="gray", linestyle=":", alpha=0.7, label="IR=50 (severe)")
ax.set_xlabel("Imbalance Ratio (log scale)", fontsize=12)
ax.set_title("Label Imbalance Ratio", fontsize=14, fontweight="bold")
ax.legend()
for i, (_, row) in enumerate(df_ir.iterrows()):
    ax.text(row["imbalance_ratio"] * 1.1, i, f"{row['imbalance_ratio']:.1f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIGS_DIR / "07_imbalance_ratio_EN.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Consolidation into JSON

In [ ]:
# 9.1 Build consolidated JSON
eda_results = {
    "metadata": {
        "notebook": "01_EDA_dataset.ipynb",
        "dataset_version_hash": dataset_hash,
        "csv_path": str(CSV_PATH),
        "imgs_dir": str(IMGS_DIR),
        "timestamp": pd.Timestamp.now().isoformat()
    },
    "integrity_audit": {
        "total_csv_rows": len(df_raw),
        "image_files_on_disk": len(img_files_on_disk),
        "unique_image_names_csv": len(img_names_csv),
        "orphan_images_on_disk": len(orphan_imgs),
        "missing_images_not_on_disk": len(missing_imgs),
        "duplicate_image_names": len(dup_names),
        "duplicate_rows_total": int(dup_mask.sum()),
        "rows_with_file_match": len(df_valid),
        "final_clean_images": len(df_clean),
        "orphan_image_list": sorted(list(orphan_imgs))[:50],
        "duplicate_name_list": sorted(list(dup_names))
    },
    "missing_values": {
        col: int(df_clean[col].isna().sum()) for col in LABEL_COLS
    },
    "label_distribution": label_stats,
    "multilabel_distribution": {
        "total_labels_per_image": df_clean["n_labels_total"].value_counts().sort_index().to_dict(),
        "pathologies_per_image": df_clean["n_pathologies"].value_counts().sort_index().to_dict()
    },
    "consistency_check": {
        "normal_only": int(normal_only),
        "alterado_only": int(alterado_only),
        "both_normal_alterado": int(both),
        "neither_normal_alterado": int(neither),
        "alterado_without_pathology": int(alterado_no_path)
    },
    "cooccurrence_pairs": df_pairs.to_dict(orient="records"),
    "artifact_pathology_interaction": df_art_path.to_dict(orient="records"),
    "suspect_pairs_audit": {
        "pairs_with_count_4": len(suspect_pairs),
        "images_with_4plus_pathologies": len(high_label_imgs),
        "images_with_6plus_pathologies": len(extreme_imgs)
    }
}

# Convert numpy types to JSON serialization
def convert_numpy(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

import json

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        elif isinstance(obj, (np.floating,)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

json_path = RESULTS_DIR / "02_eda_stats.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(eda_results, f, indent=2, ensure_ascii=False, cls=NumpyEncoder)

print(f"Results saved to: {json_path}")
print(f"File size: {json_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# 9.2 Save co-occurrence extended matrix as CSV
df_pairs.to_csv(RESULTS_DIR / "01_cooccurrence_extended.csv", index=False)
print(f"Co-occurrence CSV saved: {RESULTS_DIR / '01_cooccurrence_extended.csv'}")

# Save artifact x pathology table
df_art_path.to_csv(RESULTS_DIR / "01_artifact_pathology.csv", index=False)
print(f"Artifact x Pathology CSV saved: {RESULTS_DIR / '01_artifact_pathology.csv'}")

## 10. Executive summary

**What this notebook delivered:**

| Item | Result |
|------|----------|
| Valid images (with file + label) | Value in `01_eda_stats.json` |
| Duplicates identified and resolved | Rule: keep the line with more labels |
| Missing values ​​| Concentrates on VASCULAR ECTASIA |
| Co-occurrence pairs with metrics | PMI, nPMI, phi, lift, Jaccard |
| Suspicious pairs (co-occ = 4) audited | Probably images with 6+ labels |
| Bilingual graphics | 7 figures × 2 languages ​​= 14 files |
| Consolidated JSON | `results/01_eda_stats.json` |

**Next notebook:** `02_splits_baseline.ipynb` — iterative stratification, official splits, and M0 baseline (independent ECB).